In [76]:
from scgpt.tokenizer.gene_tokenizer import GeneVocab
from pathlib import Path
import scanpy as sc
import pandas as pd
import json
import numpy as np

In [93]:
pretrained_model_dir = Path("/local/scFM/scGPT/pretrained/scGPT_human")
path_adata = "/local/fkriegel/Moma_PD_scSAE/dat/nido_annotated.h5ad"

In [94]:
vocab_path = pretrained_model_dir / "vocab.json"
vocab = GeneVocab.from_file(vocab_path)
adata = sc.read_h5ad(path_adata)

In [95]:
dict_vocab = vocab.vocab
scgpt_vocab_genes = list(dict_vocab.keys())

In [96]:
not_in_vocab = [gene for gene in adata.var.index if gene not in scgpt_vocab_genes]

In [97]:
print(f"number of total genes in dataset:\t{len(adata.var)}")
print(f"number of genes missing from vocab:\t{len(not_in_vocab)}")

number of total genes in dataset:	36620
number of genes missing from vocab:	11814


Although there are a lot of genes missing, we will only input the hvgs into scGPT. So lets see how many of the hvgs are missing

In [98]:
mask_not_in_vocab = adata.var.index.isin(not_in_vocab)

In [99]:
adata.var

,gene_ids,feature_types,genome,canonical_feature_id,canonical_feature_id_source,mt,rb,n_cells_by_counts,mean_counts,pct_dropout_by_counts,total_counts,highly_variable,highly_variable_rank,means,variances,variances_norm,highly_variable_nbatches
MIR1302-2HG,ENSG00000243485,Gene Expression,NA,ENSG00000243485.6,ensembl_archive,False,False,42,0.000115,99.988498,42,False,NaN,0.000115,0.000115,0.517190,0
FAM138A,ENSG00000237613,Gene Expression,NA,ENSG00000237613.3,ensembl_archive,False,False,0,0.000000,100.000000,0,False,NaN,0.000000,0.000000,0.000000,0
OR4F5,ENSG00000186092,Gene Expression,NA,ENSG00000186092.7,ensembl_archive,False,False,8,0.000022,99.997809,8,False,NaN,0.000022,0.000022,0.129787,0
AL627309.1,ENSG00000238009,Gene Expression,NA,ENSG00000238009.7,ensembl_archive,False,False,6979,0.020397,98.088746,7448,False,NaN,0.020397,0.022824,0.883224,0
AL627309.3,ENSG00000239945,Gene Expression,NA,ENSG00000239945.1,ensembl_archive,False,False,63,0.000173,99.982747,63,False,NaN,0.000173,0.000173,0.598749,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
AC141272.1,ENSG00000277836,Gene Expression,NA,ENSG00000277836.1,ensembl_archive,False,False,0,0.000000,100.000000,0,False,NaN,0.000000,0.000000,0.000000,0
AC023491.2,ENSG00000278633,Gene Expression,NA,ENSG00000278633.1,ensembl_archive,False,False,38,0.000104,99.989593,38,False,NaN,0.000104,0.000104,0.495683,0
AC007325.1,ENSG00000276017,Gene Expression,NA,ENSG00000276017.1,ensembl_archive,False,False,9,0.000025,99.997535,9,False,NaN,0.000025,0.000025,0.173580,0
AC007325.4,ENSG00000278817,Gene Expression,NA,ENSG00000278817.1,ensembl_archive,False,False,9057,0.025896,97.519670,9456,False,NaN,0.025896,0.027482,0.841601,0


In [ ]:
#!/usr/bin/env python
"""
scgpt_vocab_rescue.py

Map dataset genes onto the scGPT gene vocabulary, rescuing what can be
rescued through stable identifiers.

Resolution routes, in precedence order:
  1. Ensembl ID -> suffixed vocab token.  scGPT disambiguates duplicated
     symbols by appending the Ensembl ID (e.g. "RGS5_ENSG00000143248").
     These are invisible to any symbol-level match and must be handled
     first.  They also reach loci with no HGNC record at all.
  2. Ensembl ID -> HGNC ID -> vocab token.
  3. Approved symbol -> HGNC ID -> vocab token.
  4. prev_symbol / alias_symbol -> HGNC ID -> vocab token.  alias matches
     are the only unsafe route; review them before trusting them.

Diagnostics, in the order worth reading:
  A. What kind of genes are missing (clone-based placeholders vs real
     annotated genes).  Non-exclusive buckets -- a gene can match several.
  B. Count mass lost per cell, broken down by cell type.  This is the
     quantity that matters when subset_hvg is off, and cell-type
     asymmetry here propagates into anything learned downstream.
  C. Optional HVG overlap, only meaningful if you actually intend to
     subset to HVGs.  scGPT's Preprocessor defaults subset_hvg=False.

Inputs:
  HGNC complete set (refreshed Tue/Fri, ~15 MB, note the doubled tsv/tsv/):
    https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/hgnc_complete_set.txt
  Withdrawn symbols (merged/split entries, for stubborn residual cases):
    https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/withdrawn.txt
"""

import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import anndata as ad
import scanpy as sc

# --------------------------------------------------------------------------
# config
# --------------------------------------------------------------------------
VOCAB_JSON = vocab_path
HGNC_TSV = "/local/fkriegel/Moma_PD_scSAE/dat/hgnc_complete_set.txt"
 
SYMBOL_COL = None      # adata.var column holding gene symbols; None -> var_names
ENSG_COL = "gene_ids"        # adata.var column holding Ensembl IDs, e.g. "gene_ids". Strongly recommended.
N_HVG = 3000           # the HVG budget you actually feed scGPT
OUT = Path("../res/scgpt_vocab_map.csv")

COUNTS_LAYER = "counts"    # raw integer counts
CELLTYPE_COL = "cell_type" # obs column for the loss-asymmetry breakdown
BATCH_COL = "sample_id"

RUN_HVG_DIAGNOSTIC = False # only if you plan to subset_hvg

OUT = Path("scgpt_vocab_map.csv")
SPECIAL_TOKENS = {"<pad>", "<cls>", "<eoc>", "<mask>", "<unk>", "<eos>", "<bos>"}

# --------------------------------------------------------------------------
# load
# --------------------------------------------------------------------------

sym = pd.Index(
    adata.var_names.astype(str) if SYMBOL_COL is None
    else adata.var[SYMBOL_COL].astype(str)
)
if ENSG_COL is not None and ENSG_COL in adata.var:
    ensg = (
        adata.var[ENSG_COL].astype(str)
        .str.replace(r"\.\d+$", "", regex=True)   # strip ENSG version suffix
        .str.upper().values
    )
else:
    print("WARNING: no Ensembl IDs -- routes 1 and 2 disabled, rescue will be "
          "symbol-only and far less reliable")
    ensg = np.array([None] * adata.n_vars, dtype=object)

with open(VOCAB_JSON) as fh:
    vocab_tokens = set(json.load(fh).keys()) - SPECIAL_TOKENS

# scGPT's suffixed tokens: "SYMBOL_ENSG########"
SUFFIXED = re.compile(r"^(.+)_(ENSG\d+)$")
ensg2token = {}
for t in vocab_tokens:
    mt = SUFFIXED.match(t)
    if mt:
        ensg2token[mt.group(2).upper()] = t

in_vocab = np.array([s in vocab_tokens for s in sym])
print(f"dataset genes        : {len(sym)}")
print(f"vocab tokens         : {len(vocab_tokens)} "
      f"({len(ensg2token)} ENSG-suffixed)")
print(f"direct symbol hits   : {in_vocab.sum()}")
print(f"missing              : {(~in_vocab).sum()}")

# --------------------------------------------------------------------------
# A -- triage the missing set (non-exclusive buckets)
# --------------------------------------------------------------------------
# Clone-based placeholder names are archive accession + version, e.g.
# AC004556.1, Z97192.2, U52111.1, AF279873.3.  Requiring the ".N" version
# suffix is what keeps real genes (AP2M1, U2AF1, KIF5B) out of the bucket.
PATTERNS = {
    "clone/accession placeholder (AC004556.1)": r"^[A-Z]{1,2}\d{5,6}\.\d+$",
    "LINC#####":                                r"^LINC\d+$",
    "antisense/divergent (-AS/-DT/-IT/-OT)":    r"-(AS\d*|DT|IT\d*|OT\d*)$",
    "small RNA (MIR/SNOR/SCARNA/RNU/RNVU)":     r"^(MIR|SNOR|SCARNA|RNU|RNVU|RNA5|RNY|VTRNA)",
    "pseudogene-style (-PS#/P#)":               r"(-PS\d*$|(?<=[A-Z])P\d+$)",
    "bare Ensembl ID":                          r"^ENSG\d+",
    "MT- / HLA- (format mismatch, not renamed)": r"^(MT|HLA)-",
    "make_unique artefact (-1, -2)":            r"-\d+$",
}
missing = pd.Series(sym[~in_vocab])
matched_any = pd.Series(False, index=missing.index)
print("\nA. missing-gene triage (buckets overlap):")
for label, pat in PATTERNS.items():
    hit = missing.str.match(pat, case=False)
    matched_any |= hit
    print(f"  {int(hit.sum()):>6}  {label}")
print(f"  {int((~matched_any).sum()):>6}  matched no pattern <- the only "
      f"plausibly-renameable pool")

# --------------------------------------------------------------------------
# B -- HGNC-anchored rescue
# --------------------------------------------------------------------------
hgnc = pd.read_csv(HGNC_TSV, sep="\t", dtype=str, low_memory=False)
hgnc = hgnc[hgnc["status"] == "Approved"]
approved = dict(zip(hgnc["symbol"], hgnc["hgnc_id"]))


def _long(col, key_type, upper=False):
    d = hgnc[["hgnc_id", col]].dropna(subset=[col]).copy()
    d[col] = d[col].str.split("|")
    d = d.explode(col)
    d[col] = d[col].str.strip()
    if upper:
        d[col] = d[col].str.replace(r"\.\d+$", "", regex=True).str.upper()
    d = d.rename(columns={col: "key"})
    d["key_type"] = key_type
    return d[["key", "hgnc_id", "key_type"]]


keymap = pd.concat([
    _long("ensembl_gene_id", "ensg", upper=True),
    _long("symbol", "approved"),
    _long("prev_symbol", "prev"),
    _long("alias_symbol", "alias"),
], ignore_index=True).drop_duplicates()

# Guard 1: never resolve through a prev/alias string that is the APPROVED
# symbol of a different gene -- the classic silent-reassignment path.
hijack = (
    keymap["key_type"].isin(["prev", "alias"])
    & keymap["key"].isin(approved)
    & (keymap["key"].map(approved) != keymap["hgnc_id"])
)
print(f"\ndropped {int(hijack.sum())} prev/alias keys colliding with another "
      f"gene's approved symbol")
keymap = keymap[~hijack]

# Guard 2: a key must resolve to exactly one HGNC ID.
keymap = keymap[keymap["key"].map(keymap.groupby("key")["hgnc_id"].nunique()) == 1]

PRECEDENCE = {"ensg": 0, "approved": 1, "prev": 2, "alias": 3}
keymap["rank"] = keymap["key_type"].map(PRECEDENCE)
keymap = keymap.sort_values("rank").drop_duplicates("key", keep="first")
key2id = dict(zip(keymap["key"], keymap["hgnc_id"]))
key2type = dict(zip(keymap["key"], keymap["key_type"]))


def resolve(symbol, ensembl=None):
    """Return (hgnc_id, direct_token, evidence). direct_token bypasses HGNC."""
    if ensembl and isinstance(ensembl, str):
        if ensembl in ensg2token:
            return None, ensg2token[ensembl], "ensg_suffixed"
        if ensembl in key2id:
            return key2id[ensembl], None, "ensg"
    if symbol in key2id:
        return key2id[symbol], None, key2type[symbol]
    return None, None, None


# vocab side: plain tokens resolve through HGNC; suffixed ones are handled
# by ensg2token above and deliberately skipped here.
vocab_ids = {}
for tok in vocab_tokens:
    if SUFFIXED.match(tok):
        continue
    hid, _, _ = resolve(tok)
    if hid is not None:
        vocab_ids.setdefault(hid, []).append(tok)

claimed = set(sym[in_vocab])   # tokens already taken by a direct hit

rows = []
for i, s in enumerate(sym):
    e = ensg[i]
    if in_vocab[i]:
        rows.append((s, e, None, s, "direct", "kept"))
        continue
    hid, tok, ev = resolve(s, e)
    if tok is not None:
        cands = [tok] if tok not in claimed else []
    else:
        cands = [t for t in vocab_ids.get(hid, []) if t not in claimed] if hid else []
    if len(cands) == 1:
        rows.append((s, e, hid, cands[0], ev, "rescued"))
    elif len(cands) > 1:
        rows.append((s, e, hid, "|".join(sorted(cands)), ev, "ambiguous_multi_token"))
    else:
        rows.append((s, e, hid, None, ev, "unresolved"))

m = pd.DataFrame(rows, columns=[
    "gene", "ensembl_gene_id", "hgnc_id", "vocab_token", "evidence", "status",
])
m["in_vocab"] = in_vocab

# Guard 3: two dataset rows rescued onto the same token
resc = m.index[m["status"] == "rescued"]
dup = m.loc[resc, "vocab_token"].duplicated(keep=False).to_numpy()
m.loc[resc[dup], "status"] = "ambiguous_collision"

print("\nB. rescue outcome:")
print(m["status"].value_counts().to_string())
print("\nrescued by evidence route:")
print(m.loc[m.status == "rescued", "evidence"].value_counts().to_string())
print("\nreview these manually before applying:")
for st in ["ambiguous_multi_token", "ambiguous_collision"]:
    sub = m[m.status == st]
    if len(sub):
        print(f"  {st}: {sub['gene'].tolist()}")
alias_resc = m[(m.status == "rescued") & (m.evidence == "alias")]
if len(alias_resc):
    print(f"  alias-only rescues (unsafe route): {alias_resc['gene'].tolist()}")

m.to_csv(OUT, index=False)
print(f"\nwrote {OUT}")

# --------------------------------------------------------------------------
# C -- count mass lost, by cell type.  The decision-relevant diagnostic.
# --------------------------------------------------------------------------
keep_mask = m["status"].isin(["kept", "rescued"]).to_numpy()
C = adata.layers[COUNTS_LAYER] if COUNTS_LAYER in adata.layers else adata.X

if sp.issparse(C):
    nz = C[:100].data if C.nnz else np.array([1.0])
else:
    nz = np.asarray(C[:100]).ravel()
if not np.allclose(nz, np.round(nz)):
    print("\nWARNING: counts layer is non-integer. If this matrix has been "
          "log-transformed, every variance-based ranking below (and any "
          "seurat_v3 HVG call) is fitted on the wrong scale and will fail "
          "silently rather than error.")

tot = np.asarray(C.sum(1)).ravel()
lost = np.asarray(C[:, ~keep_mask].sum(1)).ravel()
adata.obs["frac_counts_lost"] = np.where(tot > 0, lost / np.maximum(tot, 1), np.nan)

print(f"\nC. count mass lost to vocab filtering: "
      f"median {np.nanmedian(adata.obs['frac_counts_lost']):.3%}")
if CELLTYPE_COL in adata.obs:
    tab = (adata.obs.groupby(CELLTYPE_COL, observed=True)["frac_counts_lost"]
           .agg(["mean", "std", "count"]).sort_values("mean"))
    print(tab.to_string(float_format=lambda x: f"{x:.4f}"))
    spread = tab["mean"].max() - tab["mean"].min()
    print(f"\nspread across cell types: {spread:.2%} "
          f"-- a uniform loss is ignorable; a multi-point spread means the "
          f"vocab filter is a cell-type-dependent input perturbation")

# --------------------------------------------------------------------------
# D -- optional HVG overlap (only if you intend to subset_hvg)
# --------------------------------------------------------------------------
if RUN_HVG_DIAGNOSTIC:
    hv = ad.AnnData(X=C.copy(), obs=adata.obs.copy(),
                    var=pd.DataFrame(index=adata.var_names))
    sc.pp.highly_variable_genes(
        hv, n_top_genes=N_HVG, flavor="seurat_v3",
        batch_key=BATCH_COL if BATCH_COL in hv.obs else None,
    )
    is_hvg = hv.var["highly_variable"].to_numpy()
    m["is_hvg"] = is_hvg
    del hv

    lost_hvg = m["is_hvg"] & ~keep_mask
    clone = m["gene"].str.match(PATTERNS["clone/accession placeholder (AC004556.1)"])
    print(f"\nD. top-{N_HVG} HVGs not in vocab after rescue: {int(lost_hvg.sum())}")
    print(f"   of which clone/accession placeholders: {int((lost_hvg & clone).sum())}")
    print(f"   annotated genes genuinely lost: "
          f"{m.loc[lost_hvg & ~clone, 'gene'].tolist()}")
    m.to_csv(OUT, index=False)

# --------------------------------------------------------------------------
# E -- apply
# --------------------------------------------------------------------------
def apply_map(adata, m, symbol_col=None):
    """Subset to mappable genes, rename to vocab tokens, sum collapsed rows."""
    new = m["vocab_token"].where(m["status"].isin(["kept", "rescued"]))
    keep = new.notna().to_numpy()
    out = adata[:, keep].copy()
    out.var["orig_gene"] = (
        out.var_names.astype(str) if symbol_col is None
        else out.var[symbol_col].astype(str)
    ).values
    out.var_names = pd.Index(new[keep].astype(str).values)
    n_dup = int(out.var_names.duplicated().sum())
    if n_dup:
        print(f"collapsing {n_dup} duplicate token(s) by summing counts")
        out = _sum_dups(out)
    return out


def _sum_dups(a):
    """Sum rows sharing a var_name. Reorders vars alphabetically."""
    groups = pd.Categorical(a.var_names)
    ind = sp.csr_matrix(
        (np.ones(a.n_vars), (np.arange(a.n_vars), groups.codes)),
        shape=(a.n_vars, len(groups.categories)),
    )
    def mm(M):
        return M @ ind if sp.issparse(M) else np.asarray(M @ ind.toarray())
    return ad.AnnData(
        X=mm(a.X),
        obs=a.obs.copy(),
        var=pd.DataFrame(index=pd.Index(groups.categories, name=None)),
        layers={k: mm(v) for k, v in a.layers.items()},
    )



# drop the two flagged clone placeholders -> 492
m.loc[m.status.isin(["ambiguous_multi_token", "ambiguous_collision"]), "status"] = "unresolved"
m.loc[(m.status == "rescued") & (m.evidence == "alias"), "status"] = "unresolved"
m.loc[m.status == "unresolved", "vocab_token"] = None
m.to_csv(OUT, index=False)
print(m.status.value_counts().to_string())   # expect kept 24806, rescued 492
adata_v = apply_map(adata, m, SYMBOL_COL)
print(adata_v.shape)                          # expect (n_cells, 25298)
assert adata_v.var_names.isin(vocab_tokens).all()
adata_v.write_h5ad("/local/fkriegel/Moma_PD_scSAE/dat/nido_annotated.h5ad")

dataset genes        : 36620
vocab tokens         : 60694 (1363 ENSG-suffixed)
direct symbol hits   : 24806
missing              : 11814

A. missing-gene triage (buckets overlap):
   11684  clone/accession placeholder (AC004556.1)
      12  LINC#####
       0  antisense/divergent (-AS/-DT/-IT/-OT)
       0  small RNA (MIR/SNOR/SCARNA/RNU/RNVU)
       0  pseudogene-style (-PS#/P#)
       0  bare Ensembl ID
       0  MT- / HLA- (format mismatch, not renamed)
       0  make_unique artefact (-1, -2)
     118  matched no pattern <- the only plausibly-renameable pool

dropped 739 prev/alias keys colliding with another gene's approved symbol

B. rescue outcome:
status
kept                     24806
unresolved               11320
rescued                    493
ambiguous_multi_token        1

rescued by evidence route:
evidence
ensg             416
ensg_suffixed     76
alias              1

review these manually before applying:
  ambiguous_multi_token: ['AC010280.2']
  alias-only rescues (unsa

### Notes for Ilia:

- follow the preprocessing steps in the original paper for reproducability
- dont apply the `--filter_gene_by_fraction` filter in `preprocess_scgpt.py`
- in the final dataset, check how many genes are in scGPTs vocab file (adata.var). Since we are only working with 3000 hvgs as input to scGPT you only need to check the 3000 hvgs